In [1]:
import json
import os
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pricer.items import Item

load_dotenv(override=True)

True

# Choose the Data Set

In [ ]:
USERNAME = "mestico01"

DATASETS = {
    "lite": {
        "source": f"{USERNAME}/items_raw_lite",
        "target": f"{USERNAME}/items_lite",
        "expected_approx": 25_000,
    },
    "full": {
        "source": f"{USERNAME}/items_raw_full",
        "target": f"{USERNAME}/items_full",
        "expected_approx": 850_000,
    },
}
DATASET_NAME = "lite"
#DATASET_NAME = "full"
dataset_config = DATASETS[DATASET_NAME]

In [2]:
LITE_MODE = True

In [3]:
user_name = 'mestico01'
dataset = f"{user_name}/items_raw_lite" if LITE_MODE else f"{user_name}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")


display(Markdown(f"Loaded {items[0]}"))

Loaded 22,000 items


Loaded title='Odyssey K45120BLK Krom Utility/Record Case, Black' category='Musical_Instruments' price=119.95 full='Odyssey  Krom Utility/Record Case, Black\n[\'Protect your rare and hard to find records used for your special 45 rpm vinyl sets or just for safe keeping in our Krom™ series  black utility/record cases. Each has two individual compartments that hold a total of 120 7" vinyl records, 60 in each section, and can also be used as a utility case for cables and other essentials. Features include chrome plated hardware, a removable lid, a fully foam-lined interior and a heavy-duty latch and spring-loaded handle. Individual K45120 cases can be stacked on each other for convenient storage. Also available in silver ||.\']\n[\'Butterfly latch and spring-loaded handle\', \'Padlock latch holes\', \'Holds 120 7" vinyl records, 60 per compartment\', \'Foam-lined interior\', \'Rubber feet and stacking lid\']\n{"Item Weight": "7 pounds", "Product Dimensions": "9.25 x 17.75 x 9.75 inches", "Is Discontinued By Manufacturer": "No", "Date First Available": "September 6, 2012", "Color Name": "BLACK", "Material Type": "Rubber", "Brand": "Odyssey", "Color": "BLACK", "Material": "Rubber", "Style": "Utility Case for 120 7\\u2033 Vinyl Records", "Closure Type": "Toggle"}' weight=7.0 summary=None prompt=None id=None

In [4]:
items[2].id

In [5]:
for index, item in enumerate(items):
    item.id = index

In [6]:
SYSTEM_PROMPT = """
Create a concise description of a product. Respond only in this format. 
Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [7]:
display(Markdown(f"Loaded {items[0]}"))

Loaded title='Odyssey K45120BLK Krom Utility/Record Case, Black' category='Musical_Instruments' price=119.95 full='Odyssey  Krom Utility/Record Case, Black\n[\'Protect your rare and hard to find records used for your special 45 rpm vinyl sets or just for safe keeping in our Krom™ series  black utility/record cases. Each has two individual compartments that hold a total of 120 7" vinyl records, 60 in each section, and can also be used as a utility case for cables and other essentials. Features include chrome plated hardware, a removable lid, a fully foam-lined interior and a heavy-duty latch and spring-loaded handle. Individual K45120 cases can be stacked on each other for convenient storage. Also available in silver ||.\']\n[\'Butterfly latch and spring-loaded handle\', \'Padlock latch holes\', \'Holds 120 7" vinyl records, 60 per compartment\', \'Foam-lined interior\', \'Rubber feet and stacking lid\']\n{"Item Weight": "7 pounds", "Product Dimensions": "9.25 x 17.75 x 9.75 inches", "Is Discontinued By Manufacturer": "No", "Date First Available": "September 6, 2012", "Color Name": "BLACK", "Material Type": "Rubber", "Brand": "Odyssey", "Color": "BLACK", "Material": "Rubber", "Style": "Utility Case for 120 7\\u2033 Vinyl Records", "Closure Type": "Toggle"}' weight=7.0 summary=None prompt=None id=0

In [8]:
MODEL = 'gpt-6-luna'
BASE_URL = 'https://api.openai.com'
CHAT_ENDPOINT = '/v1/chat/completions'

def make_body(item):
    return {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item.full},
            ],
            "reasoning_effort": "none",
            "max_completion_tokens": 800,
    }

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise RuntimeError('Set OPENAI_API_KEY in your .env file and rerun the setup cell.')

openai_session = requests.Session()
openai_session.headers['Authorization'] = f'Bearer {api_key}'

response = openai_session.post(f'{BASE_URL}{CHAT_ENDPOINT}', json=make_body(items[0]), timeout=60)
response.raise_for_status()
preview = response.json()
display(Markdown(preview['choices'][0]['message']['content']))
print(f"Input tokens: {preview['usage']['prompt_tokens']}")
print(f"Output tokens: {preview['usage']['completion_tokens']}")

Title: Odyssey Black 7-Inch Vinyl Record and Utility Case
Category: Audio Accessories
Brand: Odyssey
Description: A black utility case designed to store and protect up to 120 7-inch vinyl records.
Details: Two foam-lined compartments hold 60 records each, with a removable lid, butterfly latch, spring-loaded handle, padlock holes, rubber feet, and a stackable design.

Input tokens: 354
Output tokens: 84


from litellm import completion

messages = [
    {
        'role': 'system',
        'content': SYSTEM_PROMPT
    },
    {
        'role': 'user',
        'content': items[0].full
    }]
response = completion(
    messages=messages,
    model='ollama/llama3.2',
    api_base='http://localhost:11434'
)
display(Markdown(response.choices[0].message.content))
print()
display(Markdown(f"Input tokens: {response.usage.prompt_tokens}"))
display(Markdown(f"Output tokens: {response.usage.completion_tokens}"))
display(Markdown(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents"))

## OpenAI Batch

Set `OPENAI_API_KEY` in your `.env` file before running the preview above. The [OpenAI Batch API](https://platform.openai.com/docs/guides/batch) processes requests asynchronously within 24 hours at a 50% discount versus synchronous requests.

Start with 10 items. Run the upload and submission cells once per job, then rerun only the last cell to check status and save available results.

In [9]:
def make_jsonl(item):
    request = {
        "custom_id": str(item.id),
        "method": "POST",
        "url": CHAT_ENDPOINT,
        "body": make_body(item),
    }
    return json.dumps(request)

def make_file(start, end, filename):
    if not (0 <= start < end <= len(items)):
        raise ValueError("Choose a valid item range")

    if end - start > 50_000:
        raise ValueError("Cannot create a file with more than 50,000 requests")
    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)

    with filename.open("w", encoding="utf-8", newline="\n") as f:
        for index in range(start, end):
            f.write(make_jsonl(items[index]) + "\n")

    max_file_size = 200 * 1024 * 1024
    if filename.stat().st_size > max_file_size:
        raise ValueError("the batch import file exceeds 200 MB.")

    return filename
    
    

In [10]:
items[0]

<Odyssey K45120BLK Krom Utility/Record Case, Black = $119.95>

In [11]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gpt-6-luna", "messages": [{"role": "system", "content": "\\nCreate a concise description of a product. Respond only in this format. \\nDo not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Odyssey  Krom Utility/Record Case, Black\\n[\'Protect your rare and hard to find records used for your special 45 rpm vinyl sets or just for safe keeping in our Krom\\u2122 series  black utility/record cases. Each has two individual compartments that hold a total of 120 7\\" vinyl records, 60 in each section, and can also be used as a utility case for cables and other essentials. Features include chrome plated hardware, a removable lid, a fully foam-lined interior and a heavy-duty latch and spring-loaded handle. Individual K45120 cases can be stacked o

In [12]:
BATCH_START = 0
BATCH_SIZE = 10
BATCH_END = min(BATCH_START + BATCH_SIZE, len(items))

batch_path = make_file(
    BATCH_START,
    BATCH_END,
    f"jsonl/openai_{BATCH_START}-{BATCH_END}.jsonl",
)

print(f"Prepared {BATCH_END - BATCH_START} requests in {batch_path}")
print(f"Input size: {batch_path.stat().st_size:,} bytes")

Prepared 10 requests in jsonl/openai_0-10.jsonl
Input size: 23,006 bytes


In [13]:
with batch_path.open("rb") as f:
    upload = openai_session.post(
        f"{BASE_URL}/v1/files",
        data={"purpose": "batch"},
        files={"file": (batch_path.name, f, "application/jsonl")},
        timeout=120,
    )

upload.raise_for_status()
input_file_id = upload.json()["id"]
print(f"Uploaded input file {input_file_id}")

Uploaded input file file-EbAhnFxqpL8tKWUH6nTKh5


In [14]:
submission = openai_session.post(
    f"{BASE_URL}/v1/batches",
    json={
        "input_file_id": input_file_id,
        "endpoint": CHAT_ENDPOINT,
        "completion_window": "24h",
    },
    timeout=60,
)

submission.raise_for_status()
batch_id = submission.json()["id"]
print(f"Batch ID: {batch_id}")

Batch ID: batch_6aff39eb00ac8190859b139d1144bee5


In [28]:
#batch_id = batch_id

status_response = openai_session.get(
    f"{BASE_URL}/v1/batches/{batch_id}",
    timeout=60,
)
status_response.raise_for_status()
batch = status_response.json()

print(f"Status: {batch['status']}")
print(f"Request counts; {batch.get('request_counts')}")

terminal_statuses = {"completed", "expired", "failed", "cancelled"}

downloaded_paths = {}

if batch["status"] not in terminal_statuses:
    print("Batch is still running. Rerun this cell later")
else:
    downloaded_paths = {}

    for kind in ("output", "error"):
        file_id = batch.get(f"{kind}_file_id")
        if not file_id:
            continue

        download = openai_session.get(
            f"{BASE_URL}/v1/files/{file_id}/content",
            stream=True,
            timeout=120,
        )
        download.raise_for_status()

        result_path = batch_path.with_name(
            f"{batch_path.stem}_{batch_id}_{kind}.jsonl"
        )

        with result_path.open("wb") as f:
            for chunk in download.iter_content(chunk_size=1024 * 1024):
                if chunk: 
                    f.write(chunk)

        downloaded_paths[kind] = result_path
        print(f"Saved {kind} to {result_path}")
        print(f"Saved {kind} file: {result_path}")



Status: completed
Request counts; {'total': 10, 'completed': 10, 'failed': 0}
Saved output to jsonl/openai_0-10_batch_6aff39eb00ac8190859b139d1144bee5_output.jsonl
Saved output file: jsonl/openai_0-10_batch_6aff39eb00ac8190859b139d1144bee5_output.jsonl


In [29]:
output_path = downloaded_paths.get("output")
error_path = downloaded_paths.get("error")

items_by_id = {str(item.id): item for item in items}
successful_ids = set()
failed_records = []


if output_path:
    with output_path.open("r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            custom_id = record.get("custom_id")
            response = record.get("response")
            error = record.get("error")

            if error or not response:
                failed_records.append(record)
                continue

            body = response.get("body", {})
            choices = body.get("choices", [])
            content = (
                choices[0].get("message", {}).get("content")
                if choices else None)
                               
            item = items_by_id.get(custom_id) 
            if item is None:
                failed_records.append(
                    {"custom_id": custom_id,
                     "error": "No matching item ID",
                     })
            elif not content:
                failed_records.append(
                    {"custom_id": custom_id,
                     "error": "Response did not contain message text",
                     })
            else: 
                item.summary = content
                successful_ids.add(custom_id)

if error_path:
    with error_path.open("r", encoding="utf-8") as f:
        for line in f:
            failed_records.append(json.loads(line))

expected_ids = {
    str(items[index].id) for index in range(BATCH_START, BATCH_END)
}   
missing_ids = expected_ids - successful_ids

print(f"Successful summaries: {len(successful_ids)}")
print(f"Failed records :{len(failed_records)}")
print(f"Missing summaries in this batch: {len(missing_ids)}")

if failed_records:
    display(failed_records[:5])

if missing_ids:
    print("First missing IDs:",sorted(missing_ids)[:10])

Successful summaries: 10
Failed records :0
Missing summaries in this batch: 0


In [30]:
print(items[0].full)

Odyssey  Krom Utility/Record Case, Black
['Protect your rare and hard to find records used for your special 45 rpm vinyl sets or just for safe keeping in our Krom™ series  black utility/record cases. Each has two individual compartments that hold a total of 120 7" vinyl records, 60 in each section, and can also be used as a utility case for cables and other essentials. Features include chrome plated hardware, a removable lid, a fully foam-lined interior and a heavy-duty latch and spring-loaded handle. Individual K45120 cases can be stacked on each other for convenient storage. Also available in silver ||.']
['Butterfly latch and spring-loaded handle', 'Padlock latch holes', 'Holds 120 7" vinyl records, 60 per compartment', 'Foam-lined interior', 'Rubber feet and stacking lid']
{"Item Weight": "7 pounds", "Product Dimensions": "9.25 x 17.75 x 9.75 inches", "Is Discontinued By Manufacturer": "No", "Date First Available": "September 6, 2012", "Color Name": "BLACK", "Material Type": "Rubbe

In [31]:
print(items[1000].summary)


None
